# Model_ENSOclim_AkaikeCoeff (February 22, 2023)

Code by Caroline Juang (c.juang@columbia.edu)

Creating a model that will perform a stepwise regression, evaluating each step using AIC (Akaike information criterion).

This model will use **ENSO to predict climate variables**

In [1]:
# import
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
import joblib # for saving models

## User input

In [2]:
# customize seasons for climate variables

# customize number of rolling periods
ant_years = 1 # antecedent years to include (2 antecedent years + current year)
ant_season = 3 # n+1 of months to include in each period (e.g. input 2 would mean 3 months)

firstyear = 1984 # first year of data
finalyear = 2022 # final year of data (should be same as burned area)
time_length = int(finalyear-firstyear+1) # get length of timeseries

# SST gradient: 
# this is the OBSERVED SST data to TRAIN the model on
# Read the climate index from the .txt file
with open('0_climindname.txt', 'r') as f:
    climindname = f.read().strip()
print(climindname +' will be used for the SST gradient')

# importing data string
directory = 'your_data_folder' # customize this
data_string = 'data//'
model_string = 'model//'+climindname+'//'

patch125-155_nino3-34 will be used for the SST gradient


In [3]:
# create strings of the types of forests, remove #5
province_names = ['1: American Semi-Desert and Desert Province',
                  '2: Arizona-New Mexico Mountains Semi-Desert-Open Woodland-Coniferous Forest-Alpine Meadow Province',
                  '3: Black Hills Coniferous Forest Province',
                  '4: California Coastal Chapparral Forest and Shrub Province',
                  '5: California Coastal Range Open Woodland-Shrub-Coniferous Forest-Meadow Province',
                  '6: California Coastal Steppe-Mixed Forest-Redwood Forest Province',
                  '7: California Dry Steppe Province',
                  '8: Cascade Mixed Forest-Coniferous Forest-Alpine Meadow Province',
                  '9: Chihuahuan Semi-Desert Province',
                  '10: Colorado Plateau Semi-Desert Province',
                  '12: Great Plains-Palouse Dry Steppe Province',
                  '13: Intermountain Semi-Desert Province',
                  '14: Intermountain Semi-Desert and Desert Province',
                  '15: Middle Rocky Mountain Steppe-Coniferous Forest-Alpine Meadow Province',
                  '16: Nevada-Utah Mountains-Semi-Desert-Coniferous Forest-Alpine Meadow Province',
                  '17: Northern Rocky Mountain Forest-Steppe-Coniferous Forest-Alpine Meadow Province',
                  '18: Pacific Lowland Mixed Forest Province',
                  '19: Sierran Steppe-Mixed Forest-Coniferous Forest-Alpine Meadow Province',
                  '20: Southern Rocky Mountain Steppe-Open Woodland-Coniferous Forest-Alpine Meadow Province',
                  '21: Southwest Plateau and Plains Dry Steppe and Shrub Province']
province_num = [item for item in range(len(province_names)+1+1)]
province_num.remove(11) # remove empty ecoregion - no overlap btwn westUS map and ecoregion
dfnames = ['allwestUS', 'ecoprov1', 'ecoprov2', 'ecoprov3', 'ecoprov4', 'ecoprov5', 
           'ecoprov6', 'ecoprov7', 'ecoprov8', 'ecoprov9', 'ecoprov10', 
           'ecoprov12','ecoprov13','ecoprov14','ecoprov15',
           'ecoprov16','ecoprov17','ecoprov18','ecoprov19','ecoprov20','ecoprov21']

## Coefficients to Exclude

From `Model_ENSOClim_CoeffCheck`, and `Model_ENSOClim_CoeffCheck-avg`

Get the start and end of ecoregions from the txt files. txt files of variables to exclude from predicting climate.

This is a correlation checker, which tells us which variables to exclude from the ENSO-climate model because the change in SST results in a change in the correlation coefficient between the variables.

In [4]:
# Model-CoeffCheck
# Get the start and end of ecoregions from the txt files. txt files of variables to exclude from predicting climate.

with open(model_string + "coeffExclude_sst_all_ecoprovinces.txt", "r") as f:
    coeffExall = [line.strip() for line in f]
f.close()
with open(model_string + "coeffExclude_sst_for_ecoprovinces.txt", "r") as f:
    coeffExfor = [line.strip() for line in f]
f.close()
with open(model_string + "coeffExclude_sst_non_ecoprovinces.txt", "r") as f:
    coeffExnon = [line.strip() for line in f]
f.close()

# get indices of where each ecoregion's outputs begin
icoeffExall = [i for i, e in 
               enumerate(coeffExall) if "+++" in e]
# get ecoregions so iteration is not manual
icoeffExfor = [i for i, e in 
               enumerate(coeffExfor) if "+++" in e]
# get ecoregions so iteration is not manual
icoeffExnon = [i for i, e in 
               enumerate(coeffExnon) if "+++" in e]
# add in last index
icoeffExall.append(len(coeffExall)+1)
icoeffExfor.append(len(coeffExfor)+1)
icoeffExnon.append(len(coeffExnon)+1)

In [5]:
# Model_CoeffCheck-avg
# (average: average of prior- and concurrent-season gSST)
# Get the start and end of ecoregions from the txt files. txt files of variables to exclude from predicting climate.

with open(model_string + "coeffExclude_sstavg_all_ecoprovinces.txt", "r") as f:
    coeffavgExall = [line.strip() for line in f]
f.close()
with open(model_string + "coeffExclude_sstavg_for_ecoprovinces.txt", "r") as f:
    coeffavgExfor = [line.strip() for line in f]
f.close()
with open(model_string + "coeffExclude_sstavg_non_ecoprovinces.txt", "r") as f:
    coeffavgExnon = [line.strip() for line in f]
f.close()

# get indices of where each ecoregion's outputs begin
icoeffavgExall = [i for i, e in 
               enumerate(coeffavgExall) if "+++" in e]
# get ecoregions so iteration is not manual
icoeffavgExfor = [i for i, e in 
               enumerate(coeffavgExfor) if "+++" in e]
# get ecoregions so iteration is not manual
icoeffavgExnon = [i for i, e in 
               enumerate(coeffavgExnon) if "+++" in e]
# add in last index
icoeffavgExall.append(len(coeffavgExall)+1)
icoeffavgExfor.append(len(coeffavgExfor)+1)
icoeffavgExnon.append(len(coeffavgExnon)+1)

In [6]:
# get the SST inputs needed
def modelinputsst_string(inputlist, varname):
    """
    Requirements: 
    inputlist = list of the Model_ENSOclim_Akaike model results (locally defined)
    ithisecoreg = the ecoregion name (globally defined)
    ithisecoregend = the next ecoregion name 
        in the list (globally defined)
    varname = the climate variable name (locally defined)
    """
    # narrow inputs list to ecoregion
    modellist = inputlist[ithisecoreg:ithisecoregend+1]
    # get all PREDICTING (where inputs start)
    iinputs = [i for i, e in 
               enumerate(modellist) if "PREDICTING" in e]
    # extract model inputs
    istart = modellist.index('PREDICTING ' + varname)
    if len(iinputs)>((iinputs.index(istart))+1): # 
        iend = iinputs[(iinputs.index(istart))+1]
        modelinputs = modellist[istart+1:iend] # range in list
    else:
        modelinputs = modellist[istart+1:] # last predictor to end of list
    return modelinputs

In [7]:
# calculate AICc for variables

def getAICc(rvalue, nyears, npredictors):
    """
    Calculate the AICc, which is the Akaike Information Criterion (AIC)
    with bias correction for small sample sizes (AICc). 
    Inputs are the variables needed for the calculation. 
    Output is a single value of the AICc.
    r = r-value calculated from a Pearson Correlation
    nyears = number of years in the timeseries
    npredictors = number of x-variables used in the model
    """
    # calculate AIC
    aic = nyears * np.log(1-rvalue**2) + 2*(npredictors+1)
    # calculate AIC with bias correction for small sample size
    aicc = aic + (2*(npredictors+1)*(npredictors+2)) / (nyears - npredictors - 2)
    return aicc

## import ecoregion data

In [8]:
# import western US study area
westUS_string = '12km//study_area//westUS.nc'
westUS = xr.open_dataset(directory + westUS_string, engine='netcdf4')
westUS = westUS.westUS

# import forest
forest_string = '12km//landcover//US_ForestType_Ruefenacht//forest_type_frac.nc'
forest = xr.open_dataset(directory + forest_string, engine='netcdf4')
forest = forest.forest_type_frac.sum(dim='ftype') # sum over all forest types

# create forest and nonforest area 
forest = westUS*forest
nonforest = westUS*(1-forest)
# add province as dimension to the westUS, to combine with ecoregions
westUS = westUS.expand_dims({'province':[0]})

In [9]:
# Bailey's ecoprovinces
ecoreg_string = '12km//landcover//Ecoregions_EPA//bailey_ecoprovince.nc'
ecoreg = xr.open_dataset(directory + ecoreg_string, engine='netcdf4')
print(ecoreg.bailey_ecoprovince.province.legend)

# add in entire western US
ecoreg = xr.concat([westUS, ecoreg.bailey_ecoprovince], dim='province')
# remove the ecoregion that does not overlap with the western US
ecoreg = ecoreg.sel(province=province_num)

1: American Semi-Desert and Desert Province, 2: Arizona-New Mexico Mountains Semi-Desert-Open Woodland-Coniferous Forest-Alpine Meadow Province, 3: Black Hills Coniferous Forest Province, 4: California Coastal Chapparral Forest and Shrub Province, 5: California Coastal Range Open Woodland-Shrub-Coniferous Forest-Meadow Province, 6: California Coastal Steppe-Mixed Forest-Redwood Forest Province, 7: California Dry Steppe Province, 8: Cascade Mixed Forest-Coniferous Forest-Alpine Meadow Province, 9: Chihuahuan Semi-Desert Province, 10: Colorado Plateau Semi-Desert Province, 11: Great Plains Steppe Province, 12: Great Plains-Palouse Dry Steppe Province, 13: Intermountain Semi-Desert Province, 14: Intermountain Semi-Desert and Desert Province, 15: Middle Rocky Mountain Steppe-Coniferous Forest-Alpine Meadow Province, 16: Nevada-Utah Mountains-Semi-Desert-Coniferous Forest-Alpine Meadow Province, 17: Northern Rocky Mountain Forest-Steppe-Coniferous Forest-Alpine Meadow Province, 18: Pacific 

In [10]:
# create strings of the types of forests, remove #5
province_names = ['1: American Semi-Desert and Desert Province',
                  '2: Arizona-New Mexico Mountains Semi-Desert-Open Woodland-Coniferous Forest-Alpine Meadow Province',
                  '3: Black Hills Coniferous Forest Province',
                  '4: California Coastal Chapparral Forest and Shrub Province',
                  '5: California Coastal Range Open Woodland-Shrub-Coniferous Forest-Meadow Province',
                  '6: California Coastal Steppe-Mixed Forest-Redwood Forest Province',
                  '7: California Dry Steppe Province',
                  '8: Cascade Mixed Forest-Coniferous Forest-Alpine Meadow Province',
                  '9: Chihuahuan Semi-Desert Province',
                  '10: Colorado Plateau Semi-Desert Province',
                  '12: Great Plains-Palouse Dry Steppe Province',
                  '13: Intermountain Semi-Desert Province',
                  '14: Intermountain Semi-Desert and Desert Province',
                  '15: Middle Rocky Mountain Steppe-Coniferous Forest-Alpine Meadow Province',
                  '16: Nevada-Utah Mountains-Semi-Desert-Coniferous Forest-Alpine Meadow Province',
                  '17: Northern Rocky Mountain Forest-Steppe-Coniferous Forest-Alpine Meadow Province',
                  '18: Pacific Lowland Mixed Forest Province',
                  '19: Sierran Steppe-Mixed Forest-Coniferous Forest-Alpine Meadow Province',
                  '20: Southern Rocky Mountain Steppe-Open Woodland-Coniferous Forest-Alpine Meadow Province',
                  '21: Southwest Plateau and Plains Dry Steppe and Shrub Province']
province_num = [item for item in range(len(province_names)+1+1)]
province_num.remove(11) # remove empty ecoregion - no overlap btwn westUS map and ecoregion
dfnames = ['allwestUS', 'ecoprov1', 'ecoprov2', 'ecoprov3', 'ecoprov4', 'ecoprov5', 
           'ecoprov6', 'ecoprov7', 'ecoprov8', 'ecoprov9', 'ecoprov10', 
           'ecoprov12','ecoprov13','ecoprov14','ecoprov15',
           'ecoprov16','ecoprov17','ecoprov18','ecoprov19','ecoprov20','ecoprov21']

# import SST gradient data
from `Data_CreateModelData`

In [11]:
# import SST gradient
climind_seasons = pd.read_csv(data_string + 'sstgrad_seasons_' +climindname+'_82_y.txt').set_index('Unnamed: 0')
climind_seasons.drop(labels=['y-1 mo 1-3', 'y-1 mo 4-6', 'y-1 mo 7-9'], axis=1, inplace=True)

# import average of prior and current-season gSST
climind_seasonsavg = pd.read_csv(data_string + 'sstgrad_seasons_' +climindname+'_82_y_avgcurr-prior.txt').set_index('Unnamed: 0')
print('import ' +climindname+'_82_y_avgcurr-prior')
climind_seasonsavg.drop(labels=['y-1 mo 1-6', 'y-1 mo 4-9'], axis=1, inplace=True)

# CHECK: n columns is the same between same-season and averaged climate indices
if len(climind_seasons.columns) == len(climind_seasonsavg.columns):
    print('number of seasons present is correct')
else:
    raise RuntimeError("Number of seasons columns is mismatched. Stopping the kernel.")

import patch125-155_nino3-34_82_y_avgcurr-prior
number of seasons present is correct


# Import climate data
from `Data_CreateModelData`

In [12]:
# import model data for climate

dfframesall = {}
dfframesfor = {}
dfframesnon = {}

for iecoreg in np.arange(len(province_num)):
    filename = data_string + 'climatefull_ecoprovinces_'
    dfframesall[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_all.txt').set_index('Unnamed: 0')
    dfframesfor[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_for.txt').set_index('Unnamed: 0')
    dfframesnon[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_non.txt').set_index('Unnamed: 0')

# Model

## Correlation and regression
5/11/2022
1. Regress all potential predictors, one at a time, against log10(burned area) and get regression coefficients
2. Use each pair regression coefficients, one at a time, to estimate log10(burned area)
3. Calculate correlation between estimated and observed records of log10(burned area)
4. Calculate the Akaike Information Criterion (AIC) with the bias correction for small sample size (AICc) for each correlation coefficient.
5. Select the regression model that leads to the most negative AICc value and save this most negative AICc value.
6. For each of the remaining potential predictors, do a multiple regression where each remaining predictor is used along with the initial best predictor to estimate log10(burned area). For each remaining predictor, calculate the new correlation between estimated and observed log10(burned area), and calculate the AICc associated with each estimated time series (accounting for the increase in number of predictors in the AIC calculation).
7. Is the new lowest AICc value lower than the previous lowest AICc value by more than 2? If so, update your model using the two variables in a multiple regression and repeat step 6. If not, the stepwise process has failed to find any more useful predictor variables, so finalize your regression model using only the predictors identified prior to this latest step.
8. If the AICc is not a negative value, exclude the variable from consideration. (new 3/15/2023)
9. If no variables are included because they all have positive AICc values, then set the regression coefficient of the variable to zero and the intercept to the average of the predictor climate value.

Akaike Information Criterion - https://doi.org/10.1109/TAC.1974.1100705

```AIC = Nyears * log(1-r.^2) + 2*(Npredictors+1)```

* `r^2` = square of the correlation coefficient
* `Nyears` = number of years in the timeseries
* `Npredictors` = the number of timeseries you're using as predictors
* rewarded for large sample size, penalized for adding more number of predictors

regression and time series model selection in small samples: https://doi.org/10.1093/biomet/76.2.297

```AICc = AIC + (2*(Npredictors+1)*(Npredictors+2))/(Nyears - Npredictors - 2)```
* bias correction on the AIC

8/26/2024

We also include the files created from `Model_ENSOclim_CoeffCheck` and `Model_ENSOclim_CoeffCheck-avg`, which are used to exclude SST gradient variables that might be correlated with climate for the wrong reasons. If the correlation sign changes between SST and climate, then we exclude the variable for consideration.

**The final result looks like this:**
1. **Use the concurrent-season gSST to predict climate (e.g. January-March gSST), OR**
2. **Use the average of prior-season and concurrent-season gSST to predict climate (e.g. October-March gSST)**

The logic of the following code is this:
1. Linear regression of climate with concurrent or average(prior-season + concurrent-season) gSST
2. Calculate AICc for both
3. For each, see if the variable needs to be excluded according to `Model_ENSOclim_CoeffCheck` and `Model_ENSOclim_CoeffCheck-avg`. If so, set the calculated AICc to `99`.
4. Find the minimum AICc.
5. Use the gSST variable with the most negative AICc as the final model.
6. If AICc is 99, then set the regression coefficient to `0` and set the intercept to the mean value. (If AICc is positive, then it is still ok to be included in the model)

In [13]:
# storage for each model's variable indices
dfmodelall = {}
dfmodelfor = {}
dfmodelnon = {}

# Run model for ALL

In [14]:
# regress all potential predictors against climate, for ALL area

# for the coefficient to exclude
landtypeinput = coeffExall
landtypeiinput = icoeffExall
landtypeavginput = coeffavgExall
landtypeavgiinput = icoeffavgExall

print('ALL LANDCOVER')
land = 'all'
modeloutputfile = model_string + 'modeloutput_sst_all_'+land+'.txt'
f = open(modeloutputfile, 'w') # print to this file
labels = dfframesall['allwestUS'].columns.values # each climate variable
for iregion in np.arange(len(province_num)): # iterate through ecoregion

    print('++++++++ ALL LANDCOVER model for ' + dfnames[iregion]+ '\t AICC \t r-value \tCoeff Case \tAICc Case \tCoefficient++++++')
    f.write('+++'+dfnames[iregion]+'\n')
    for i, label in enumerate(labels): # iterate through climate variables
        tmpaicc = [] # empty list of aicc values for each regression fit
        tmpi = [] # empty list of indices for variables

        # retrieve coefficients to exclude
        ithisecoreg = landtypeinput.index('+++'+dfnames[iregion])
        ithisecoregend = landtypeiinput[landtypeiinput.index(ithisecoreg)+1]-1
        modelexclude = modelinputsst_string(landtypeinput, label)
        
        ithisecoreg = landtypeavginput.index('+++'+dfnames[iregion])
        ithisecoregend = landtypeavgiinput[landtypeavgiinput.index(ithisecoreg)+1]-1
        modelexcludeavg = modelinputsst_string(landtypeavginput, label)

        # VARIABLES TO COMPARE
        # each climate variable
        tmpclim = np.asarray(dfframesall[dfnames[iregion]][label]).reshape(-1,1)
        # associated concurrent-season climate index (ENSO or other)
        strseason = label.split(' ',1)[1] # get season label
        matchseason = climind_seasons.columns.isin([strseason]) # True/False array
        tmpclimind = np.asarray(climind_seasons.loc[:,matchseason])
        imatchseason = np.where(matchseason)[0]
        tmpclimindavg = np.asarray(climind_seasonsavg.iloc[:,imatchseason])
        # grab average of (previous season+current season)         
        iprevseason = int(imatchseason)-1
        tmpclimindprev = np.asarray(climind_seasons.iloc[:,iprevseason]).reshape(-1,1)
        strprevseason = climind_seasons.columns[iprevseason] # label

        print('PREDICTING '+label)
        f.write('PREDICTING '+label)
        f.write('\n')

        # STEP 1 linear regression, compare concurrent, or prior gSST
        sstlist = [tmpclimind, tmpclimindavg]
        aicc = np.zeros(len(sstlist))
        
        for i,selectclimind in enumerate(sstlist):
            reg = LinearRegression() # least squares
            reg.fit(selectclimind, tmpclim)
            tmppredict = reg.predict(selectclimind)
            tmpr = pearsonr(tmpclim.flatten(), tmppredict.flatten())[0]

            # calculate AIC
            nyears = len(selectclimind)
            npredictors = reg.n_features_in_
            tmpgetaicc = getAICc(tmpr, nyears, npredictors)
            
            # store AICC and r-value
            aicc[i] = tmpgetaicc

        # check if (1) current-season correlation is in CoeffExclude
        if (len(modelexclude)==1) & (strseason in modelexclude):
            aicc[0] = 99 # set to massive positive if so
            print(climind_seasonsavg.columns[imatchseason].tolist()[0] + '\t' + str(np.round(aicc[1], decimals=3))
                 + '\tCASE 2 EXCL CONCURR')
        # check if (2) avgcurrent+prior is in CoeffExclude
        if (len(modelexcludeavg)==1) & (strseason in modelexcludeavg):
            aicc[1] = 99
            print(climind_seasons.columns[matchseason].tolist()[0] + '\t' + str(np.round(aicc[0], decimals=3))
                  + '\tCASE 3 EXCL AVG')
        
        # get minimum of the two values (iminaicc would be 0 or 1)
        [iminaicc, minaicc] = np.argmin(aicc), np.min(aicc)
        tmpi.append(iminaicc) # add min to list of variables
        tmpaicc.append(minaicc) # add aicc value to list
        
        # regress model again
        reg = LinearRegression() # least squares
        reg.fit(sstlist[iminaicc], tmpclim)
            
        if minaicc>98: # as long as CoeffExclude not used # new, 8/23/2024
            # force change the coefficient and intercept
            reg.coef_ = np.asarray([[0]])
            reg.intercept_ = tmpclim.mean()
            print('\tAICC POSITIVE--CHANGE COEF' 
                  + '\tCoef ='+str(np.round(reg.coef_, decimals=3)))
            # store the modified model
            filename = model_string + 'model_'+land+'_'+label+'_'+dfnames[iregion]+'.sav' # rewrite model and save
            joblib.dump(reg, filename)
        else: # if there is a negative min AICC, that is the new model
            # store the newest model
            filename = model_string + 'model_'+land+'_'+label+'_'+dfnames[iregion]+'.sav' # rewrite model and save
            joblib.dump(reg, filename)
        
        # record the season (in column name) to the file
        if (iminaicc==0):
            f.write(climind_seasons.columns[matchseason].tolist()[0])
            f.write('\n')
            print(climind_seasons.columns[matchseason].tolist()[0] + '\t' + str(np.round(minaicc, decimals=3)) + 
              '\t' + str(np.round(tmpr, decimals=3)) + '\tFINAL MODEL v1'
             + '\tCoef ='+str(np.round(reg.coef_, decimals=3)))
        elif (iminaicc==1):
            f.write(climind_seasonsavg.columns[imatchseason].tolist()[0])
            f.write('\n')
            print(climind_seasonsavg.columns[imatchseason].tolist()[0] + '\t' + str(np.round(minaicc, decimals=3)) + 
              '\t' + str(np.round(tmpr, decimals=3)) + '\tFINAL MODEL v2'
             + '\tCoef ='+str(np.round(reg.coef_, decimals=3)))

    dfmodelall[dfnames[iregion]] = [labels[i] for i in tmpi] # add this ecoregion's values to list
f.close()

ALL LANDCOVER
++++++++ ALL LANDCOVER model for allwestUS	 AICC 	 r-value 	Coeff Case 	AICc Case 	Coefficient++++++
PREDICTING rh y0 mo 1-3
y0 mo 1-3	-12.405	0.523	FINAL MODEL v1	Coef =[[-0.546]]
PREDICTING rh y0 mo 4-6
y0 mo 1-6	-11.708	0.575	FINAL MODEL v2	Coef =[[-0.685]]
PREDICTING rh y0 mo 7-9
y0 mo 4-9	0.803	CASE 2 EXCL CONCURR
y0 mo 7-9	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 7-9	99.0	0.29	FINAL MODEL v1	Coef =[[0]]
PREDICTING rh y0 mo 10-12
y0 mo 10-12	1.032	CASE 3 EXCL AVG
y0 mo 10-12	1.032	0.28	FINAL MODEL v1	Coef =[[-0.22]]
PREDICTING solar y0 mo 1-3
y0 mo 1-3	-5.432	0.413	FINAL MODEL v1	Coef =[[0.434]]
PREDICTING solar y0 mo 4-6
y0 mo 1-6	-2.181	CASE 2 EXCL CONCURR
y0 mo 1-6	-2.181	0.387	FINAL MODEL v2	Coef =[[0.462]]
PREDICTING solar y0 mo 7-9
y0 mo 7-9	0.758	CASE 3 EXCL AVG
y0 mo 7-9	0.758	0.274	FINAL MODEL v1	Coef =[[0.278]]
PREDICTING solar y0 mo 10-12
y0 mo 10-12	4.323	CASE 3 EXCL AVG
y0 mo 10-12	4.323	0.038	FINAL MODEL v1	Coef =[[-0.004]]
PRE

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = 

y0 mo 1-3	-16.981	0.604	FINAL MODEL v1	Coef =[[0.6]]
PREDICTING vpd y0 mo 4-6
y0 mo 1-6	-13.263	0.596	FINAL MODEL v2	Coef =[[0.711]]
PREDICTING vpd y0 mo 7-9
y0 mo 7-9	4.037	CASE 3 EXCL AVG
y0 mo 7-9	4.037	0.062	FINAL MODEL v1	Coef =[[0.081]]
PREDICTING vpd y0 mo 10-12
y0 mo 10-12	-5.387	0.443	FINAL MODEL v1	Coef =[[0.363]]
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	-9.963	0.52	FINAL MODEL v1	Coef =[[-0.512]]
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	2.672	CASE 3 EXCL AVG
y0 mo 4-6	2.672	0.231	FINAL MODEL v1	Coef =[[-0.262]]
PREDICTING wetdays y0 mo 7-9
y0 mo 4-9	3.289	CASE 2 EXCL CONCURR
y0 mo 7-9	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 7-9	99.0	0.16	FINAL MODEL v1	Coef =[[0]]
PREDICTING wetdays y0 mo 10-12
y0 mo 7-12	-6.392	0.485	FINAL MODEL v2	Coef =[[-0.425]]
PREDICTING wind y0 mo 1-3
y-1 mo 10-3	-0.838	0.348	FINAL MODEL v2	Coef =[[0.293]]
PREDICTING wind y0 mo 4-6
y0 mo 4-6	1.031	CASE 3 EXCL AVG
y0 mo 4-6	1.031	0.29	FINAL MODEL v1	Coef =[[0.367]]
PREDICTING win

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = 

y0 mo 4-6	4.197	0.151	FINAL MODEL v1	Coef =[[-0.074]]
PREDICTING rh y0 mo 7-9
y0 mo 7-9	4.323	CASE 3 EXCL AVG
y0 mo 7-9	4.323	0.05	FINAL MODEL v1	Coef =[[0.005]]
PREDICTING rh y0 mo 10-12
y0 mo 10-12	3.572	CASE 3 EXCL AVG
y0 mo 10-12	3.572	0.088	FINAL MODEL v1	Coef =[[0.107]]
PREDICTING solar y0 mo 1-3
y0 mo 1-3	-1.276	0.316	FINAL MODEL v1	Coef =[[0.338]]
PREDICTING solar y0 mo 4-6
y0 mo 1-6	1.948	CASE 2 EXCL CONCURR
y0 mo 4-6	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 4-6	99.0	0.24	FINAL MODEL v1	Coef =[[0]]
PREDICTING solar y0 mo 7-9
y0 mo 7-9	4.146	CASE 3 EXCL AVG
y0 mo 7-9	4.146	0.051	FINAL MODEL v1	Coef =[[0.063]]
PREDICTING solar y0 mo 10-12
y0 mo 10-12	3.296	CASE 3 EXCL AVG
y0 mo 10-12	3.296	0.079	FINAL MODEL v1	Coef =[[-0.125]]
PREDICTING tmax y0 mo 1-3
y0 mo 1-3	4.317	CASE 3 EXCL AVG
y0 mo 1-3	4.317	0.049	FINAL MODEL v1	Coef =[[0.013]]
PREDICTING tmax y0 mo 4-6
y0 mo 4-6	4.324	CASE 3 EXCL AVG
y0 mo 4-6	4.324	0.123	FINAL MODEL v1	Coef =[[0.003]]
PREDICTI

...

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_35536/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_35536/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_35536/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = 

y0 mo 4-6	99.0	0.213	FINAL MODEL v1	Coef =[[0]]
PREDICTING prec y0 mo 7-9
y0 mo 7-9	4.064	CASE 3 EXCL AVG
y0 mo 7-9	4.064	0.071	FINAL MODEL v1	Coef =[[-0.077]]
PREDICTING prec y0 mo 10-12
y0 mo 10-12	3.554	CASE 3 EXCL AVG
y0 mo 10-12	3.554	0.076	FINAL MODEL v1	Coef =[[0.108]]
++++++++ ALL LANDCOVER model for ecoprov20	 AICC 	 r-value 	Coeff Case 	AICc Case 	Coefficient++++++
PREDICTING rh y0 mo 1-3
y0 mo 1-3	-5.313	0.407	FINAL MODEL v1	Coef =[[-0.432]]
PREDICTING rh y0 mo 4-6
y0 mo 1-6	-7.933	0.514	FINAL MODEL v2	Coef =[[-0.612]]
PREDICTING rh y0 mo 7-9
y0 mo 7-9	1.635	CASE 3 EXCL AVG
y0 mo 7-9	1.635	0.301	FINAL MODEL v1	Coef =[[-0.243]]
PREDICTING rh y0 mo 10-12
y0 mo 10-12	1.109	CASE 3 EXCL AVG
y0 mo 10-12	1.109	0.266	FINAL MODEL v1	Coef =[[-0.217]]
PREDICTING solar y0 mo 1-3
y0 mo 1-3	-3.024	0.375	FINAL MODEL v1	Coef =[[0.383]]
PREDICTING solar y0 mo 4-6
y0 mo 1-6	-2.031	0.383	FINAL MODEL v2	Coef =[[0.457]]
PREDICTING solar y0 mo 7-9
y0 mo 7-9	2.513	CASE 3 EXCL AVG
y0 mo 7-9	2.513	0

y0 mo 10-12	-8.284	0.51	FINAL MODEL v1	Coef =[[0.407]]
PREDICTING tmin y0 mo 1-3
y0 mo 1-3	3.708	CASE 3 EXCL AVG
y0 mo 1-3	3.708	0.133	FINAL MODEL v1	Coef =[[-0.115]]
PREDICTING tmin y0 mo 4-6
y0 mo 1-6	-11.457	0.571	FINAL MODEL v2	Coef =[[0.681]]
PREDICTING tmin y0 mo 7-9
y0 mo 7-9	3.102	CASE 3 EXCL AVG
y0 mo 7-9	3.102	0.158	FINAL MODEL v1	Coef =[[0.165]]
PREDICTING tmin y0 mo 10-12
y0 mo 10-12	1.82	CASE 3 EXCL AVG
y0 mo 10-12	1.82	0.253	FINAL MODEL v1	Coef =[[0.193]]
PREDICTING tmean y0 mo 1-3
y0 mo 1-3	1.668	CASE 3 EXCL AVG
y0 mo 1-3	1.668	0.231	FINAL MODEL v1	Coef =[[0.237]]
PREDICTING tmean y0 mo 4-6
y0 mo 1-6	-10.024	0.549	FINAL MODEL v2	Coef =[[0.654]]
PREDICTING tmean y0 mo 7-9
y0 mo 7-9	3.325	CASE 3 EXCL AVG
y0 mo 7-9	3.325	0.111	FINAL MODEL v1	Coef =[[0.15]]
PREDICTING tmean y0 mo 10-12
y0 mo 10-12	-6.141	0.475	FINAL MODEL v1	Coef =[[0.375]]
PREDICTING vpd y0 mo 1-3
y0 mo 1-3	-12.481	0.567	FINAL MODEL v1	Coef =[[0.547]]
PREDICTING vpd y0 mo 4-6
y0 mo 1-6	-4.209	0.438	FINAL MO

y0 mo 10-12	4.041	0.042	FINAL MODEL v1	Coef =[[0.066]]
PREDICTING prec y0 mo 1-3
y0 mo 1-3	-0.023	0.309	FINAL MODEL v1	Coef =[[-0.3]]
PREDICTING prec y0 mo 4-6
y0 mo 4-6	3.916	CASE 3 EXCL AVG
y0 mo 4-6	3.916	0.093	FINAL MODEL v1	Coef =[[-0.131]]
PREDICTING prec y0 mo 7-9
y0 mo 7-9	3.528	CASE 3 EXCL AVG
y0 mo 7-9	3.528	0.095	FINAL MODEL v1	Coef =[[-0.134]]
PREDICTING prec y0 mo 10-12
y0 mo 10-12	-2.672	0.388	FINAL MODEL v1	Coef =[[-0.313]]


/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_35536/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_35536/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_35536/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = 

# Run Model for FOREST

In [15]:
# regress all potential predictors against climate, for FOREST area
print('FOREST')
land = 'for'

# for the coefficient to exclude
landtypeinput = coeffExfor
landtypeiinput = icoeffExfor
landtypeavginput = coeffavgExfor
landtypeavgiinput = icoeffavgExfor

modeloutputfile = model_string + 'modeloutput_sst_'+land+'.txt'
f = open(modeloutputfile, 'w') # print to this file
labels = dfframesfor['allwestUS'].columns.values # each climate variable
for iregion in np.arange(len(province_num)): # iterate through ecoregion

    print('++++++++ FOREST model for ' + dfnames[iregion]+ '\t AICC \t r-value \tCoeff Case \tAICc Case \tCoefficient++++++')
    f.write('+++'+dfnames[iregion]+'\n')
    for i, label in enumerate(labels): # iterate through climate variables
        tmpaicc = [] # empty list of aicc values for each regression fit
        tmpi = [] # empty list of indices for variables

        # retrieve coefficients to exclude
        ithisecoreg = landtypeinput.index('+++'+dfnames[iregion])
        ithisecoregend = landtypeiinput[landtypeiinput.index(ithisecoreg)+1]-1
        modelexclude = modelinputsst_string(landtypeinput, label)

        ithisecoreg = landtypeavginput.index('+++'+dfnames[iregion])
        ithisecoregend = landtypeavgiinput[landtypeavgiinput.index(ithisecoreg)+1]-1
        modelexcludeavg = modelinputsst_string(landtypeavginput, label)

        # VARIABLES TO COMPARE
        # each climate variable
        tmpclim = np.asarray(dfframesfor[dfnames[iregion]][label]).reshape(-1,1)
        # associated concurrent-season climate index (ENSO or other)
        strseason = label.split(' ',1)[1] # get season label
        matchseason = climind_seasons.columns.isin([strseason]) # True/False array
        tmpclimind = np.asarray(climind_seasons.loc[:,matchseason])
        imatchseason = np.where(matchseason)[0]
        tmpclimindavg = np.asarray(climind_seasonsavg.iloc[:,imatchseason])
        # grab average of (previous season+current season)         
        iprevseason = int(imatchseason)-1
        tmpclimindprev = np.asarray(climind_seasons.iloc[:,iprevseason]).reshape(-1,1)
        strprevseason = climind_seasons.columns[iprevseason] # label

        print('PREDICTING '+label)
        f.write('PREDICTING '+label)
        f.write('\n')

        # STEP 1 linear regression, compare concurrent, or prior gSST
        sstlist = [tmpclimind, tmpclimindavg]
        aicc = np.zeros(len(sstlist))
        
        for i,selectclimind in enumerate(sstlist):
            reg = LinearRegression() # least squares
            reg.fit(selectclimind, tmpclim)
            tmppredict = reg.predict(selectclimind)
            tmpr = pearsonr(tmpclim.flatten(), tmppredict.flatten())[0]
            # calculate AIC
            nyears = len(selectclimind)
            npredictors = reg.n_features_in_
            tmpgetaicc = getAICc(tmpr, nyears, npredictors)

            # store AICC and r-value
            aicc[i] = tmpgetaicc

        # check if (1) current-season correlation is in CoeffExclude
        if (len(modelexclude)==1) & (strseason in modelexclude):
            aicc[0] = 99 # set to massive positive if so
            print(climind_seasonsavg.columns[imatchseason].tolist()[0] + '\t' + str(np.round(aicc[1], decimals=3))
                 + '\tCASE 2 EXCL CONCURR')
        # check if (2) avgcurrent+prior is in CoeffExclude
        if (len(modelexcludeavg)==1) & (strseason in modelexcludeavg):
            aicc[1] = 99
            print(climind_seasons.columns[matchseason].tolist()[0] + '\t' + str(np.round(aicc[0], decimals=3))
                  + '\tCASE 3 EXCL AVG')
        
        # get minimum of the two values (iminaicc would be 0 or 1)
        [iminaicc, minaicc] = np.argmin(aicc), np.min(aicc)
        tmpi.append(iminaicc) # add min to list of variables
        tmpaicc.append(minaicc) # add aicc value to list
        
        # regress model again
        reg = LinearRegression() # least squares
        reg.fit(sstlist[iminaicc], tmpclim)
            
        if minaicc>98: # as long as CoeffExclude not used # new, 8/23/2024
            # force change the coefficient and intercept
            reg.coef_ = np.asarray([[0]])
            reg.intercept_ = tmpclim.mean()
            print('\tAICC POSITIVE--CHANGE COEF' 
                  + '\tCoef ='+str(np.round(reg.coef_, decimals=3)))
            # store the modified model
            filename = model_string + 'model_'+land+'_'+label+'_'+dfnames[iregion]+'.sav' # rewrite model and save
            joblib.dump(reg, filename)
        else: # if there is a negative min AICC, that is the new model
            # store the newest model
            filename = model_string + 'model_'+land+'_'+label+'_'+dfnames[iregion]+'.sav' # rewrite model and save
            joblib.dump(reg, filename)
        
        # record the season (in column name) to the file
        if (iminaicc==0):
            f.write(climind_seasons.columns[matchseason].tolist()[0])
            f.write('\n')
            print(climind_seasons.columns[matchseason].tolist()[0] + '\t' + str(np.round(minaicc, decimals=3)) + 
              '\t' + str(np.round(tmpr, decimals=3)) + '\tFINAL MODEL v1'
             + '\tCoef ='+str(np.round(reg.coef_, decimals=3)))
        elif (iminaicc==1):
            f.write(climind_seasonsavg.columns[imatchseason].tolist()[0])
            f.write('\n')
            print(climind_seasonsavg.columns[imatchseason].tolist()[0] + '\t' + str(np.round(minaicc, decimals=3)) + 
              '\t' + str(np.round(tmpr, decimals=3)) + '\tFINAL MODEL v2'
             + '\tCoef ='+str(np.round(reg.coef_, decimals=3)))

    dfmodelfor[dfnames[iregion]] = [labels[i] for i in tmpi] # add this ecoregion's values to list
f.close()

FOREST
++++++++ FOREST model for allwestUS	 AICC 	 r-value 	Coeff Case 	AICc Case 	Coefficient++++++
PREDICTING rh y0 mo 1-3
y0 mo 1-3	-11.129	0.493	FINAL MODEL v1	Coef =[[-0.529]]
PREDICTING rh y0 mo 4-6
y0 mo 1-6	-7.806	0.511	FINAL MODEL v2	Coef =[[-0.61]]
PREDICTING rh y0 mo 7-9
y0 mo 4-9	0.937	CASE 2 EXCL CONCURR
y0 mo 7-9	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 7-9	99.0	0.285	FINAL MODEL v1	Coef =[[0]]
PREDICTING rh y0 mo 10-12
y0 mo 10-12	2.566	CASE 3 EXCL AVG
y0 mo 10-12	2.566	0.207	FINAL MODEL v1	Coef =[[-0.162]]
PREDICTING solar y0 mo 1-3
y0 mo 1-3	0.131	CASE 3 EXCL AVG
y0 mo 1-3	0.131	0.267	FINAL MODEL v1	Coef =[[0.295]]
PREDICTING solar y0 mo 4-6
y0 mo 1-6	0.983	CASE 2 EXCL CONCURR
y0 mo 4-6	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 4-6	99.0	0.283	FINAL MODEL v1	Coef =[[0]]
PREDICTING solar y0 mo 7-9
y0 mo 7-9	1.295	CASE 3 EXCL AVG
y0 mo 7-9	1.295	0.271	FINAL MODEL v1	Coef =[[0.257]]
PREDICTING solar y0 mo 10-12
y0 mo 10-12

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/53734470.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/53734470.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/53734470.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(im

y0 mo 1-3	-16.217	0.59	FINAL MODEL v1	Coef =[[0.592]]
PREDICTING vpd y0 mo 4-6
y0 mo 1-6	-14.947	0.618	FINAL MODEL v2	Coef =[[0.737]]
PREDICTING vpd y0 mo 7-9
y0 mo 7-9	4.164	CASE 3 EXCL AVG
y0 mo 7-9	4.164	0.047	FINAL MODEL v1	Coef =[[0.06]]
PREDICTING vpd y0 mo 10-12
y0 mo 10-12	-4.666	0.427	FINAL MODEL v1	Coef =[[0.351]]
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	-10.906	0.532	FINAL MODEL v1	Coef =[[-0.525]]
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	2.614	CASE 3 EXCL AVG
y0 mo 4-6	2.614	0.248	FINAL MODEL v1	Coef =[[-0.267]]
PREDICTING wetdays y0 mo 7-9
y0 mo 4-9	2.446	CASE 2 EXCL CONCURR
y0 mo 7-9	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 7-9	99.0	0.214	FINAL MODEL v1	Coef =[[0]]
PREDICTING wetdays y0 mo 10-12
y0 mo 7-12	-6.191	0.481	FINAL MODEL v2	Coef =[[-0.421]]
PREDICTING wind y0 mo 1-3
y-1 mo 10-3	-1.629	0.372	FINAL MODEL v2	Coef =[[0.313]]
PREDICTING wind y0 mo 4-6
y0 mo 4-6	1.321	CASE 3 EXCL AVG
y0 mo 4-6	1.321	0.275	FINAL MODEL v1	Coef =[[0.351]]
PREDICTING

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/53734470.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/53734470.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/53734470.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(im

y0 mo 4-6	1.884	0.117	FINAL MODEL v1	Coef =[[0.317]]
PREDICTING vpd y0 mo 7-9
y0 mo 4-9	-1.952	CASE 2 EXCL CONCURR
y0 mo 4-9	-1.952	0.381	FINAL MODEL v2	Coef =[[0.454]]
PREDICTING vpd y0 mo 10-12
y0 mo 10-12	2.56	CASE 3 EXCL AVG
y0 mo 10-12	2.56	0.167	FINAL MODEL v1	Coef =[[0.162]]
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	4.28	CASE 3 EXCL AVG
y0 mo 1-3	4.28	0.051	FINAL MODEL v1	Coef =[[-0.031]]
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	3.955	CASE 3 EXCL AVG
y0 mo 4-6	3.955	0.052	FINAL MODEL v1	Coef =[[-0.125]]
PREDICTING wetdays y0 mo 7-9
y0 mo 4-9	0.492	CASE 2 EXCL CONCURR
y0 mo 4-9	0.492	0.302	FINAL MODEL v2	Coef =[[-0.36]]
PREDICTING wetdays y0 mo 10-12
y0 mo 10-12	3.425	CASE 3 EXCL AVG
y0 mo 10-12	3.425	0.168	FINAL MODEL v1	Coef =[[0.117]]
PREDICTING wind y0 mo 1-3
y0 mo 1-3	-5.902	0.421	FINAL MODEL v1	Coef =[[0.444]]
PREDICTING wind y0 mo 4-6
y0 mo 1-6	-5.353	0.464	FINAL MODEL v2	Coef =[[0.553]]
PREDICTING wind y0 mo 7-9
y0 mo 7-9	4.324	CASE 3 EXCL AVG
y0 mo 7-9	4.324	0.007	FINAL MODEL

...

y0 mo 1-3	4.136	CASE 3 EXCL AVG
y0 mo 1-3	4.136	0.006	FINAL MODEL v1	Coef =[[0.064]]
PREDICTING tmax y0 mo 4-6
y0 mo 4-6	4.112	CASE 3 EXCL AVG
y0 mo 4-6	4.112	0.194	FINAL MODEL v1	Coef =[[0.095]]
PREDICTING tmax y0 mo 7-9
y0 mo 7-9	2.522	CASE 3 EXCL AVG
y0 mo 7-9	2.522	0.27	FINAL MODEL v1	Coef =[[0.2]]
PREDICTING tmax y0 mo 10-12
y0 mo 10-12	4.143	CASE 3 EXCL AVG
y0 mo 10-12	4.143	0.096	FINAL MODEL v1	Coef =[[0.053]]
PREDICTING tmin y0 mo 1-3
y-1 mo 10-3	-0.494	0.337	FINAL MODEL v2	Coef =[[-0.284]]
PREDICTING tmin y0 mo 4-6
y0 mo 4-6	4.14	CASE 3 EXCL AVG
y0 mo 4-6	4.14	0.022	FINAL MODEL v1	Coef =[[-0.088]]
PREDICTING tmin y0 mo 7-9
y0 mo 7-9	2.325	CASE 3 EXCL AVG
y0 mo 7-9	2.325	0.252	FINAL MODEL v1	Coef =[[0.21]]
PREDICTING tmin y0 mo 10-12
y0 mo 10-12	3.799	CASE 3 EXCL AVG
y0 mo 10-12	3.799	0.086	FINAL MODEL v1	Coef =[[0.089]]
PREDICTING tmean y0 mo 1-3
y0 mo 1-3	4.012	CASE 3 EXCL AVG
y0 mo 1-3	4.012	0.155	FINAL MODEL v1	Coef =[[-0.082]]
PREDICTING tmean y0 mo 4-6
y0 mo 4-6	4.304	CAS

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/53734470.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/53734470.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/53734470.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(im

y0 mo 10-12	-3.863	0.409	FINAL MODEL v1	Coef =[[-0.337]]


y0 mo 10-12	-1.807	0.359	FINAL MODEL v1	Coef =[[0.295]]
PREDICTING vpd y0 mo 1-3
y0 mo 1-3	-0.186	CASE 3 EXCL AVG
y0 mo 1-3	-0.186	0.274	FINAL MODEL v1	Coef =[[0.305]]
PREDICTING vpd y0 mo 4-6
y0 mo 1-6	-5.549	0.468	FINAL MODEL v2	Coef =[[0.557]]
PREDICTING vpd y0 mo 7-9
y0 mo 4-9	-1.834	CASE 2 EXCL CONCURR
y0 mo 4-9	-1.834	0.378	FINAL MODEL v2	Coef =[[0.45]]
PREDICTING vpd y0 mo 10-12
y0 mo 10-12	-1.1	0.335	FINAL MODEL v1	Coef =[[0.279]]
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	4.242	CASE 3 EXCL AVG
y0 mo 1-3	4.242	0.01	FINAL MODEL v1	Coef =[[-0.042]]
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	-0.928	0.335	FINAL MODEL v1	Coef =[[-0.458]]
PREDICTING wetdays y0 mo 7-9
y0 mo 7-9	3.608	CASE 3 EXCL AVG
y0 mo 7-9	3.608	0.115	FINAL MODEL v1	Coef =[[-0.127]]
PREDICTING wetdays y0 mo 10-12
y0 mo 10-12	4.307	CASE 3 EXCL AVG
y0 mo 10-12	4.307	0.048	FINAL MODEL v1	Coef =[[-0.016]]
PREDICTING wind y0 mo 1-3
y0 mo 1-3	-6.137	0.472	FINAL MODEL v1	Coef =[[0.448]]
PREDICTING wind y0 mo 4-6
y0 mo 1-6	-6.001	

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_35536/53734470.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_35536/53734470.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_35536/53734470.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(im

# Run Model for NONFOREST

In [16]:
# regress all potential predictors against climate, for NONFOREST area

print('NONFOREST')
land = 'non'

# for the coefficient to exclude
landtypeinput = coeffExnon
landtypeiinput = icoeffExnon
landtypeavginput = coeffavgExnon
landtypeavgiinput = icoeffavgExnon

modeloutputfile = model_string + 'modeloutput_sst_'+land+'.txt'
f = open(modeloutputfile, 'w') # print to this file
labels = dfframesnon['allwestUS'].columns.values # each climate variable
for iregion in np.arange(len(province_num)): # iterate through ecoregion

    print('++++++++ NONFOREST model for ' + dfnames[iregion]+ '\t AICC \t r-value \tCoeff Case \tAICc Case \tCoefficient++++++')
    f.write('+++'+dfnames[iregion]+'\n')
    for i, label in enumerate(labels): # iterate through climate variables
        tmpaicc = [] # empty list of aicc values for each regression fit
        tmpi = [] # empty list of indices for variables

        # retrieve coefficients to exclude
        ithisecoreg = landtypeinput.index('+++'+dfnames[iregion])
        ithisecoregend = landtypeiinput[landtypeiinput.index(ithisecoreg)+1]-1
        modelexclude = modelinputsst_string(landtypeinput, label)

        ithisecoreg = landtypeavginput.index('+++'+dfnames[iregion])
        ithisecoregend = landtypeavgiinput[landtypeavgiinput.index(ithisecoreg)+1]-1
        modelexcludeavg = modelinputsst_string(landtypeavginput, label)

        # VARIABLES TO COMPARE
        # each climate variable
        tmpclim = np.asarray(dfframesnon[dfnames[iregion]][label]).reshape(-1,1)
        # associated concurrent-season climate index (ENSO or other)
        strseason = label.split(' ',1)[1] # get season label
        matchseason = climind_seasons.columns.isin([strseason]) # True/False array
        tmpclimind = np.asarray(climind_seasons.loc[:,matchseason])
        imatchseason = np.where(matchseason)[0]
        tmpclimindavg = np.asarray(climind_seasonsavg.iloc[:,imatchseason])
        # grab average of (previous season+current season)         
        iprevseason = int(imatchseason)-1
        tmpclimindprev = np.asarray(climind_seasons.iloc[:,iprevseason]).reshape(-1,1)
        strprevseason = climind_seasons.columns[iprevseason] # label

        print('PREDICTING '+label)
        f.write('PREDICTING '+label)
        f.write('\n')

        # STEP 1 linear regression, compare concurrent, or prior gSST
        sstlist = [tmpclimind, tmpclimindavg]
        aicc = np.zeros(len(sstlist))
        
        for i,selectclimind in enumerate(sstlist):
            reg = LinearRegression() # least squares
            reg.fit(selectclimind, tmpclim)
            tmppredict = reg.predict(selectclimind)
            tmpr = pearsonr(tmpclim.flatten(), tmppredict.flatten())[0]

            # calculate AIC
            nyears = len(selectclimind)
            npredictors = reg.n_features_in_
            tmpgetaicc = getAICc(tmpr, nyears, npredictors)

            # store AICC and r-value
            aicc[i] = tmpgetaicc

        # check if (1) current-season correlation is in CoeffExclude
        if (len(modelexclude)==1) & (strseason in modelexclude):
            aicc[0] = 99 # set to massive positive if so
            print(climind_seasonsavg.columns[imatchseason].tolist()[0] + '\t' + str(np.round(aicc[1], decimals=3))
                 + '\tCASE 2 EXCL CONCURR')
        # check if (2) avgcurrent+prior is in CoeffExclude
        if (len(modelexcludeavg)==1) & (strseason in modelexcludeavg):
            aicc[1] = 99
            print(climind_seasons.columns[matchseason].tolist()[0] + '\t' + str(np.round(aicc[0], decimals=3))
                  + '\tCASE 3 EXCL AVG')
        
        # get minimum of the two values (iminaicc would be 0 or 1)
        [iminaicc, minaicc] = np.argmin(aicc), np.min(aicc)
        tmpi.append(iminaicc) # add min to list of variables
        tmpaicc.append(minaicc) # add aicc value to list
        
        # regress model again
        reg = LinearRegression() # least squares
        reg.fit(sstlist[iminaicc], tmpclim)
            
        if minaicc>98: # as long as CoeffExclude not used # new, 8/23/2024
            # force change the coefficient and intercept
            reg.coef_ = np.asarray([[0]])
            reg.intercept_ = tmpclim.mean()
            print('\tAICC POSITIVE--CHANGE COEF' 
                  + '\tCoef ='+str(np.round(reg.coef_, decimals=3)))
            # store the modified model
            filename = model_string + 'model_'+land+'_'+label+'_'+dfnames[iregion]+'.sav' # rewrite model and save
            joblib.dump(reg, filename)
        else: # if there is a negative min AICC, that is the new model
            # store the newest model
            filename = model_string + 'model_'+land+'_'+label+'_'+dfnames[iregion]+'.sav' # rewrite model and save
            joblib.dump(reg, filename)
        
        # record the season (in column name) to the file
        if (iminaicc==0):
            f.write(climind_seasons.columns[matchseason].tolist()[0])
            f.write('\n')
            print(climind_seasons.columns[matchseason].tolist()[0] + '\t' + str(np.round(minaicc, decimals=3)) + 
              '\t' + str(np.round(tmpr, decimals=3)) + '\tFINAL MODEL v1'
             + '\tCoef ='+str(np.round(reg.coef_, decimals=3)))
        elif (iminaicc==1):
            f.write(climind_seasonsavg.columns[imatchseason].tolist()[0])
            f.write('\n')
            print(climind_seasonsavg.columns[imatchseason].tolist()[0] + '\t' + str(np.round(minaicc, decimals=3)) + 
              '\t' + str(np.round(tmpr, decimals=3)) + '\tFINAL MODEL v2'
             + '\tCoef ='+str(np.round(reg.coef_, decimals=3)))
    
    dfmodelnon[dfnames[iregion]] = [labels[i] for i in tmpi] # add this ecoregion's values to list

NONFOREST
++++++++ NONFOREST model for allwestUS	 AICC 	 r-value 	Coeff Case 	AICc Case 	Coefficient++++++
PREDICTING rh y0 mo 1-3
y0 mo 1-3	-11.945	0.522	FINAL MODEL v1	Coef =[[-0.54]]
PREDICTING rh y0 mo 4-6
y0 mo 1-6	-12.745	0.589	FINAL MODEL v2	Coef =[[-0.703]]
PREDICTING rh y0 mo 7-9
y0 mo 7-9	2.501	CASE 3 EXCL AVG
y0 mo 7-9	2.501	0.289	FINAL MODEL v1	Coef =[[-0.201]]
PREDICTING rh y0 mo 10-12
y0 mo 10-12	0.497	CASE 3 EXCL AVG
y0 mo 10-12	0.497	0.301	FINAL MODEL v1	Coef =[[-0.236]]
PREDICTING solar y0 mo 1-3
y0 mo 1-3	-7.845	0.461	FINAL MODEL v1	Coef =[[0.478]]
PREDICTING solar y0 mo 4-6
y0 mo 1-6	-3.097	0.411	FINAL MODEL v2	Coef =[[0.491]]
PREDICTING solar y0 mo 7-9
y0 mo 7-9	0.757	CASE 3 EXCL AVG
y0 mo 7-9	0.757	0.266	FINAL MODEL v1	Coef =[[0.278]]
PREDICTING solar y0 mo 10-12
y0 mo 10-12	4.16	CASE 3 EXCL AVG
y0 mo 10-12	4.16	0.103	FINAL MODEL v1	Coef =[[0.05]]
PREDICTING tmax y0 mo 1-3
y0 mo 1-3	4.264	CASE 3 EXCL AVG
y0 mo 1-3	4.264	0.023	FINAL MODEL v1	Coef =[[0.036]]
PREDICTI

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/4266087664.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/4266087664.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/4266087664.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = 

y0 mo 10-12	3.805	CASE 3 EXCL AVG
y0 mo 10-12	3.805	0.15	FINAL MODEL v1	Coef =[[-0.089]]
PREDICTING wind y0 mo 1-3
y-1 mo 10-3	-2.079	0.385	FINAL MODEL v2	Coef =[[0.324]]
PREDICTING wind y0 mo 4-6
y0 mo 1-6	-5.284	0.462	FINAL MODEL v2	Coef =[[0.551]]
PREDICTING wind y0 mo 7-9
y0 mo 7-9	4.242	CASE 3 EXCL AVG
y0 mo 7-9	4.242	0.001	FINAL MODEL v1	Coef =[[0.043]]
PREDICTING wind y0 mo 10-12
y0 mo 10-12	2.282	CASE 3 EXCL AVG
y0 mo 10-12	2.282	0.219	FINAL MODEL v1	Coef =[[0.175]]
PREDICTING prec y0 mo 1-3
y0 mo 1-3	-6.402	0.424	FINAL MODEL v1	Coef =[[-0.453]]
PREDICTING prec y0 mo 4-6
y0 mo 1-6	-0.487	CASE 2 EXCL CONCURR
y0 mo 1-6	-0.487	0.337	FINAL MODEL v2	Coef =[[-0.401]]
PREDICTING prec y0 mo 7-9
y0 mo 7-9	2.761	CASE 3 EXCL AVG
y0 mo 7-9	2.761	0.222	FINAL MODEL v1	Coef =[[-0.186]]
PREDICTING prec y0 mo 10-12
y0 mo 10-12	4.258	CASE 3 EXCL AVG
y0 mo 10-12	4.258	0.077	FINAL MODEL v1	Coef =[[-0.032]]
++++++++ NONFOREST model for ecoprov1	 AICC 	 r-value 	Coeff Case 	AICc Case 	Coefficient+++

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/4266087664.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/4266087664.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/4266087664.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = 

y0 mo 7-9	2.745	0.21	FINAL MODEL v1	Coef =[[0.187]]
PREDICTING tmin y0 mo 10-12
y0 mo 10-12	4.125	CASE 3 EXCL AVG
y0 mo 10-12	4.125	0.07	FINAL MODEL v1	Coef =[[-0.055]]
PREDICTING tmean y0 mo 1-3
y-1 mo 10-3	0.292	0.31	FINAL MODEL v2	Coef =[[-0.261]]
PREDICTING tmean y0 mo 4-6
y0 mo 4-6	4.101	CASE 3 EXCL AVG
y0 mo 4-6	4.101	0.026	FINAL MODEL v1	Coef =[[0.097]]
PREDICTING tmean y0 mo 7-9
y0 mo 7-9	1.012	CASE 3 EXCL AVG
y0 mo 7-9	1.012	0.315	FINAL MODEL v1	Coef =[[0.268]]
PREDICTING tmean y0 mo 10-12
y0 mo 10-12	4.312	CASE 3 EXCL AVG
y0 mo 10-12	4.312	0.03	FINAL MODEL v1	Coef =[[-0.014]]
PREDICTING vpd y0 mo 1-3
y0 mo 1-3	4.32	CASE 3 EXCL AVG
y0 mo 1-3	4.32	0.05	FINAL MODEL v1	Coef =[[-0.01]]
PREDICTING vpd y0 mo 4-6
y0 mo 4-6	2.393	CASE 3 EXCL AVG
y0 mo 4-6	2.393	0.086	FINAL MODEL v1	Coef =[[0.283]]
PREDICTING vpd y0 mo 7-9
y0 mo 4-9	-1.108	CASE 2 EXCL CONCURR
y0 mo 4-9	-1.108	0.356	FINAL MODEL v2	Coef =[[0.425]]
PREDICTING vpd y0 mo 10-12
y0 mo 10-12	2.84	CASE 3 EXCL AVG
y0 mo 10-12	2.

...

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/4266087664.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/4266087664.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/4266087664.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = 

y0 mo 7-9	4.323	0.098	FINAL MODEL v1	Coef =[[0.006]]
PREDICTING vpd y0 mo 10-12
y0 mo 7-12	-0.549	0.339	FINAL MODEL v2	Coef =[[-0.297]]
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	4.281	CASE 3 EXCL AVG
y0 mo 1-3	4.281	0.02	FINAL MODEL v1	Coef =[[-0.031]]
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	4.292	CASE 3 EXCL AVG
y0 mo 4-6	4.292	0.045	FINAL MODEL v1	Coef =[[0.037]]
PREDICTING wetdays y0 mo 7-9
y0 mo 7-9	3.162	CASE 3 EXCL AVG
y0 mo 7-9	3.162	0.189	FINAL MODEL v1	Coef =[[-0.161]]
PREDICTING wetdays y0 mo 10-12
y0 mo 10-12	2.674	CASE 3 EXCL AVG
y0 mo 10-12	2.674	0.197	FINAL MODEL v1	Coef =[[0.157]]
PREDICTING wind y0 mo 1-3
y0 mo 1-3	4.323	CASE 3 EXCL AVG
y0 mo 1-3	4.323	0.017	FINAL MODEL v1	Coef =[[-0.006]]
PREDICTING wind y0 mo 4-6
y0 mo 1-6	1.266	CASE 2 EXCL CONCURR
y0 mo 4-6	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 4-6	99.0	0.271	FINAL MODEL v1	Coef =[[0]]
PREDICTING wind y0 mo 7-9
y0 mo 7-9	4.233	CASE 3 EXCL AVG
y0 mo 7-9	4.233	0.081	FINAL MODEL v1	Coef =[[0.045

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/4266087664.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/4266087664.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/4266087664.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = 

y0 mo 10-12	3.73	0.116	FINAL MODEL v1	Coef =[[0.095]]
PREDICTING vpd y0 mo 1-3
y0 mo 1-3	0.339	CASE 3 EXCL AVG
y0 mo 1-3	0.339	0.238	FINAL MODEL v1	Coef =[[0.288]]
PREDICTING vpd y0 mo 4-6
y0 mo 1-6	-1.499	CASE 2 EXCL CONCURR
y0 mo 1-6	-1.499	0.368	FINAL MODEL v2	Coef =[[0.439]]
PREDICTING vpd y0 mo 7-9
y0 mo 4-9	0.549	CASE 2 EXCL CONCURR
y0 mo 7-9	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 7-9	99.0	0.3	FINAL MODEL v1	Coef =[[0]]
PREDICTING vpd y0 mo 10-12
y0 mo 10-12	4.178	CASE 3 EXCL AVG
y0 mo 10-12	4.178	0.101	FINAL MODEL v1	Coef =[[0.047]]
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	-1.185	0.31	FINAL MODEL v1	Coef =[[-0.335]]
PREDICTING wetdays y0 mo 4-6
y0 mo 1-6	-0.05	CASE 2 EXCL CONCURR
y0 mo 1-6	-0.05	0.322	FINAL MODEL v2	Coef =[[-0.384]]
PREDICTING wetdays y0 mo 7-9
y0 mo 7-9	4.047	CASE 3 EXCL AVG
y0 mo 7-9	4.047	0.083	FINAL MODEL v1	Coef =[[-0.079]]
PREDICTING wetdays y0 mo 10-12
y0 mo 10-12	4.268	CASE 3 EXCL AVG
y0 mo 10-12	4.268	0.029	FINAL MODEL v1	Coef 

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/4266087664.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/4266087664.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_36323/4266087664.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = 

In [17]:
# close files
f.close()
# print last time this model was updated
from datetime import datetime
now = datetime.now()
# dd/mm/YY H:M:S
dt_string = now.strftime("%d/%m/%Y %H:%M:%S")
print('SST gradient used: '+climindname)
print("Model last run =", dt_string)

SST gradient used: patch125-155_nino3-34
Model last run = 18/11/2025 11:48:56
